### Import Libraries and Reproducibility

In [ ]:
import sys, os, re, json, pandas as pd, numpy as np
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
import transformers

SEED = 22
transformers.set_seed(SEED)
rng = np.random.default_rng(SEED)

### Configuration and Instructions

In [ ]:
TRAIN_PATH = "train_w2.csv"
TEST_PATH  = "test_w2.csv"
LABEL      = "SNAP"
OUTPUT_DIR = "./results_wave2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

# ── Feature sets ────────────────────────────────────────────────────────
# SNAP_wave1 is a LONGITUDINAL predictor: self-reported Wave 1 SNAP
# participation carried forward into the Wave 2 feature set.

DEMO_FEATURES     = [
    "marital.status", "sex", "hispanic.origin", "race",
    "education", "citizenship", "employment", "state", "age",
]
INCOME_FEATURES   = DEMO_FEATURES + ["income"]
LONGITUDINAL_FEATURES = ["SNAP_wave1"]
FULL_FEATURES     = INCOME_FEATURES + LONGITUDINAL_FEATURES          # canonical Wave 2 full model
MINIMAL_FEATURES  = ["income", "employment", "education", "SNAP_wave1"]

COL_MAP = {
    "marital.status":  "Marital Status",
    "sex":             "Sex",
    "hispanic.origin": "Hispanic Origin",
    "race":            "Race",
    "education":       "Educational Attainment",
    "citizenship":     "Citizenship",
    "employment":      "Employment Status",
    "state":           "State of Residence",
    "age":             "Age",
    "income":          "Income (Wave 2)",
    "SNAP_wave1":      "SNAP Coverage in Wave 1 (Prior Wave)",
    "SNAP":            "Coverage for SNAP in Wave 2",
}

# ── Instructions ────────────────────────────────────────────────────────
INSTRUCTION_STRUCTURED = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food-purchasing "
    "assistance to low-income individuals and families in the U.S. "
    "You will be given a respondent's demographic information and income from "
    "Wave 2 of the 2014 Survey of Income and Program Participation (SIPP), "
    "along with whether they had SNAP coverage in Wave 1 (the prior survey wave). "
    "This Wave 1 SNAP status captures prior "
    "program participation and benefit receipt for that individual. "
    "Based on all this information, classify if the person has SNAP coverage in Wave 2. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

INSTRUCTION_NARRATIVE = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food assistance "
    "to low-income households in the U.S. "
    "You will read a short profile of a survey respondent from the 2014 Survey of "
    "Income and Program Participation (SIPP), Wave 2. "
    "The profile includes whether the respondent received SNAP in Wave 1 (the "
    "immediately preceding survey wave). "
    "Based on the full profile, decide whether this person has SNAP coverage in Wave 2. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

INSTRUCTION_MINIMAL = INSTRUCTION_STRUCTURED

### Data Loading

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {len(df_train):,} rows | SNAP distribution: {dict(Counter(df_train[LABEL]))}")
print(f"Test  : {len(df_test):,} rows  | SNAP distribution: {dict(Counter(df_test[LABEL]))}")

# Sanity-check that the longitudinal predictor is present
assert "SNAP_wave1" in df_train.columns, "SNAP_wave1 column missing from train_w2.csv"
assert "SNAP_wave1" in df_test.columns,  "SNAP_wave1 column missing from test_w2.csv"
print("\nSNAP_wave1 confirmed in both splits.")
print(f"Train SNAP_wave1 dist : {dict(Counter(df_train['SNAP_wave1']))}")
print(f"Test  SNAP_wave1 dist : {dict(Counter(df_test['SNAP_wave1']))}")

Train : 27,311 rows | SNAP distribution: {'No': 23948, 'Yes': 3363}
Test  : 11,703 rows  | SNAP distribution: {'No': 10263, 'Yes': 1440}

SNAP_wave1 confirmed in both splits.
Train SNAP_wave1 dist : {'No': 24165, 'Yes': 3146}
Test  SNAP_wave1 dist : {'No': 10349, 'Yes': 1354}


### Oversampling

In [ ]:
def oversample_minority(df, label_col, seed=SEED):
    majority = df[df[label_col] == "No"]
    minority = df[df[label_col] == "Yes"]
    minority_upsampled = minority.sample(
        n=len(majority), replace=True, random_state=seed
    )
    return pd.concat([majority, minority_upsampled]).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

df_train_balanced = oversample_minority(df_train, LABEL)
print(f"\nBalanced train: {len(df_train_balanced):,} rows | "
      f"SNAP distribution: {dict(Counter(df_train_balanced[LABEL]))}")


Balanced train: 47,896 rows | SNAP distribution: {'No': 23948, 'Yes': 23948}


### Prompt Builders

In [ ]:
def build_structured_prompt(row, features, instruction=INSTRUCTION_STRUCTURED):
    user_content = "\n".join(
        f"{COL_MAP[k]}: {row[k]}" for k in features if k in row
    )
    return {
        "prompt":     [{"role": "system",    "content": instruction},
                       {"role": "user",      "content": user_content}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }


def build_narrative_prompt(row, features, instruction=INSTRUCTION_NARRATIVE):
    age        = row.get("age", "unknown age")
    sex        = row.get("sex", "").lower()
    hispanic   = row.get("hispanic.origin", "")
    race       = row.get("race", "")
    ethnicity  = f"{hispanic} {race}".strip()
    marital    = row.get("marital.status", "").lower()
    edu        = row.get("education", "").lower()
    employment = row.get("employment", "").lower()
    state      = row.get("state", "")
    income     = row.get("income", "")
    citizenship = row.get("citizenship", "").lower()
    snap_w1    = row.get("SNAP_wave1", "Unknown")

    narrative = (
        f"A {age}-year-old {ethnicity} {sex}, {marital}, residing in {state}. "
        f"Educational attainment: {edu}. "
        f"Citizenship: {citizenship}. "
        f"Employment in the reference month: {employment}. "
        f"Monthly household income (Wave 2): {income}. "
        f"SNAP coverage in Wave 1 (prior wave): {snap_w1}."
    )
    return {
        "prompt":     [{"role": "system",    "content": instruction},
                       {"role": "user",      "content": narrative}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }


def make_dataset(df, prompt_fn, features):
    from datasets import Dataset
    records = [prompt_fn(row, features) for _, row in df.iterrows()]
    return Dataset.from_list(records)

### Export to JSONL (MLX Format)

In [ ]:
def export_to_jsonl(df, prompt_fn, features, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            sample   = prompt_fn(row, features)
            messages = sample["prompt"] + sample["completion"]
            f.write(json.dumps({"messages": messages}) + "\n")


# Experiment matrix — same as Wave 1 structure
# T2A uses INCOME_FEATURES (demographics + income, no longitudinal predictor)
# T2C_longitudinal_model uses only SNAP_wave1
experiments_data = {
    "T1A_imbalanced":   (df_train,          build_structured_prompt, FULL_FEATURES,     FULL_FEATURES),
    "T1B_oversampled":  (df_train_balanced,  build_structured_prompt, FULL_FEATURES,     FULL_FEATURES),
    "T2A_demo_only":    (df_train,          build_structured_prompt, INCOME_FEATURES,     INCOME_FEATURES),
    "T2C_longitudinal_model":   (df_train,          build_structured_prompt, LONGITUDINAL_FEATURES,     LONGITUDINAL_FEATURES),
    "T3A_narrative":    (df_train,          build_narrative_prompt,  FULL_FEATURES,     FULL_FEATURES),
    "T3B_minimal":      (df_train,          build_structured_prompt, MINIMAL_FEATURES,  MINIMAL_FEATURES),
}

os.makedirs("./mlx_data", exist_ok=True)
for run_label, (train_df, prompt_fn, train_feats, val_feats) in experiments_data.items():
    folder = f"./mlx_data/{run_label}"
    os.makedirs(folder, exist_ok=True)
    export_to_jsonl(train_df, prompt_fn, train_feats, f"{folder}/train.jsonl")
    export_to_jsonl(df_test,  prompt_fn, val_feats,   f"{folder}/valid.jsonl")
    print(f"Exported {folder}/")

Exported ./mlx_data/T1A_imbalanced/
Exported ./mlx_data/T1B_oversampled/
Exported ./mlx_data/T2A_demo_only/
Exported ./mlx_data/T2C_full_model/
Exported ./mlx_data/T3A_narrative/
Exported ./mlx_data/T3B_minimal/


### Making Test Datasets

In [ ]:
test_ds_structured_full     = make_dataset(df_test, build_structured_prompt, FULL_FEATURES)
test_ds_structured_demo     = make_dataset(df_test, build_structured_prompt, INCOME_FEATURES)
test_ds_structured_longitudinal   = make_dataset(df_test, build_structured_prompt, LONGITUDINAL_FEATURES)
test_ds_narrative_full      = make_dataset(df_test, build_narrative_prompt,  FULL_FEATURES)
test_ds_structured_minimal  = make_dataset(df_test, build_structured_prompt, MINIMAL_FEATURES)
print("Test datasets ready.")

Test datasets ready.


### Defining Predictions

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

def evaluate_predictions(predictions, references, label_name=""):
    valid         = [(p, r) for p, r in zip(predictions, references) if p in ["Yes", "No"]]
    unknown_count = len(predictions) - len(valid)
    if not valid:
        print("No valid predictions.")
        return {}
    y_pred, y_true = zip(*valid)
    metrics = {
        "Label":     label_name,
        "N_test":    len(df_test),
        "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "Recall":    round(recall_score(y_true, y_pred, pos_label="Yes"), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label="Yes", zero_division=0), 4),
        "F1":        round(f1_score(y_true, y_pred, pos_label="Yes"), 4),
        "Unknown":   unknown_count,
    }
    print(f"\n── {label_name} ──")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return metrics

### Calibration Analytics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

def calibration_analysis(probabilities, references, label_name, save_path):
    y_true_bin = [1 if r == "Yes" else 0 for r in references]
    probs      = np.array(probabilities)

    n_bins    = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece       = 0.0
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = np.mean(np.array(y_true_bin)[mask])
        bin_conf = np.mean(probs[mask])
        ece += mask.sum() / len(probs) * abs(bin_acc - bin_conf)

    frac_pos, mean_pred = calibration_curve(y_true_bin, probs, n_bins=10)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(mean_pred, frac_pos, "s-", label=f"{label_name} (ECE={ece:.4f})")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(f"Reliability Diagram — {label_name}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  ECE: {ece:.4f} | Saved → {save_path}")
    return round(ece, 4)

### Run Inference Function

In [ ]:
from mlx_lm import load, generate

def format_chat_prompt(prompt_list):
    return "".join(
        f"<|{m['role']}|>\n{m['content']}\n" for m in prompt_list
    ) + "<|assistant|>\n"


def run_inference_mlx(model, tokenizer, dataset, return_probs=False):
    predictions, references, probabilities = [], [], []

    for example in tqdm(dataset, desc="Generating"):
        prompt_text = format_chat_prompt(example["prompt"])
        response    = generate(
            model, tokenizer,
            prompt=prompt_text,
            max_tokens=5,
            verbose=False
        )
        match = re.search(r"\b(Yes|No)\b", response, re.IGNORECASE)
        predictions.append(match.group(1).title() if match else "Unknown")
        references.append(example["completion"][0]["content"].strip().title())

        if return_probs:
            if match:
                probabilities.append(1.0 if match.group(1).title() == "Yes" else 0.0)
            else:
                probabilities.append(0.5)

    if return_probs:
        return predictions, references, probabilities
    return predictions, references

### LoRA Configuration

In [ ]:
import yaml

lora_config = {
    "lora_parameters": {
        "rank":    16,
        "scale":   2.0,   # equivalent to alpha=32 with rank=16 (scale = alpha/rank)
        "dropout": 0.05,
    }
}

os.makedirs("./configs", exist_ok=True)
with open("./configs/lora_config.yaml", "w") as f:
    yaml.dump(lora_config, f)

print("LoRA config saved to ./configs/lora_config.yaml")

LoRA config saved to ./configs/lora_config.yaml


### Training and Inference for T1A_imbalanced

In [ ]:
!mlx_lm.lora \
  --model {MODEL_ID} \
  --train \
  --data ./mlx_data/T1A_imbalanced \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w2/T1A_imbalanced \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 85019.68it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:21<00:00,  1.16it/s]
Iter 1: Val loss 7.781, Val took 21.585s
Iter 10: Train loss 4.049, Learning Rate 2.000e-04, It/sec 0.589, Tokens/sec 9.420, Trained Tokens 160, Peak mem 5.730 GB
Iter 20: Train loss 0.172, Learning Rate 2.000e-04, It/sec 0.611, Tokens/sec 9.773, Trained Tokens 320, Peak mem 5.730 GB
Iter 30: Train loss 0.090, Learning Rate 2.000e-04, It/sec 0.608, Tokens/sec 9.731, Trained Tokens 480, Peak mem 5.730 GB
Iter 40: Train loss 0.557, Learning Rate 2.000e-04, It/sec 0.608, Tokens/sec 9.724, Trained Tokens 640, Peak mem 5.730 GB
Iter 50: Train loss 0.113, Learning Rate 2.000e-04, It/sec 0.610, 

In [ ]:
import time
# ── Initialise accumulators on first experiment ──────────────────
all_metrics      = []
experiment_times = {}

# ── Load model with T1A_imbalanced adapter ──────────────────────────────
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w2/T1A_imbalanced",
)

# ── Inference ──────────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

# ── Evaluate ───────────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T1A_imbalanced")

# ── Calibration ────────────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T1A_imbalanced",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T1A_imbalanced.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ───────────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1A_imbalanced.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1A_imbalanced"] = elapsed
print(f"\n  ⏱ T1A_imbalanced done in {elapsed:.1f}s")

# ── Clean up ───────────────────────────────────────────────────────
del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11703 [00:00<?, ?it/s]


── T1A_imbalanced ──
  Label: T1A_imbalanced
  N_test: 11703
  Accuracy: 0.8876
  Recall: 0.1306
  Precision: 0.746
  F1: 0.2222
  Unknown: 0
              precision    recall  f1-score   support

          No       0.89      0.99      0.94     10263
         Yes       0.75      0.13      0.22      1440

    accuracy                           0.89     11703
   macro avg       0.82      0.56      0.58     11703
weighted avg       0.87      0.89      0.85     11703

  ECE: 0.1070 | Saved → ./results_wave2/calibration_T1A_imbalanced.png

  ⏱ T1A_imbalanced done in 9778.6s


### Training and Inference for T1B_oversampled

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T1B_oversampled \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w2/T1B_oversampled \
    --save-every 500 \
    -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 14682.51it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:21<00:00,  1.16it/s]
Iter 1: Val loss 7.788, Val took 21.473s
Iter 10: Train loss 4.352, Learning Rate 2.000e-04, It/sec 0.605, Tokens/sec 9.681, Trained Tokens 160, Peak mem 5.730 GB
Iter 20: Train loss 0.452, Learning Rate 2.000e-04, It/sec 0.614, Tokens/sec 9.830, Trained Tokens 320, Peak mem 5.730 GB
Iter 30: Train loss 0.269, Learning Rate 2.000e-04, It/sec 0.614, Tokens/sec 9.830, Trained Tokens 480, Peak mem 5.730 GB
Iter 40: Train loss 0.482, Learning Rate 2.000e-04, It/sec 0.616, Tokens/sec 9.850, Trained Tokens 640, Peak mem 5.730 GB
Iter 50: Train loss 0.264, Learning Rate 2.000e-04, It/sec 0.614, 

In [ ]:
# ── Load model with T1B_oversampled adapter ──────────────────────────────
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w2/T1B_oversampled",
)

# ── Inference ──────────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

# ── Evaluate ───────────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T1B_oversampled")

# ── Calibration ────────────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T1B_oversampled",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T1B_oversampled.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ───────────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1B_oversampled.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1B_oversampled"] = elapsed
print(f"\n  ⏱ T1B_oversampled done in {elapsed:.1f}s")

# ── Clean up ───────────────────────────────────────────────────────
del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11703 [00:00<?, ?it/s]


── T1B_oversampled ──
  Label: T1B_oversampled
  N_test: 11703
  Accuracy: 0.8629
  Recall: 0.6451
  Precision: 0.4594
  F1: 0.5367
  Unknown: 0
              precision    recall  f1-score   support

          No       0.95      0.89      0.92     10263
         Yes       0.46      0.65      0.54      1440

    accuracy                           0.86     11703
   macro avg       0.70      0.77      0.73     11703
weighted avg       0.89      0.86      0.87     11703

  ECE: 0.0437 | Saved → ./results_wave2/calibration_T1B_oversampled.png

  ⏱ T1B_oversampled done in 3753.1s


### Training and Inference for T2A_demo_only

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T2A_demo_only \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w2/T2A_demo_only \
    --save-every 500 \
    -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 17508.69it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:21<00:00,  1.18it/s]
Iter 1: Val loss 7.781, Val took 21.241s
Iter 10: Train loss 4.049, Learning Rate 2.000e-04, It/sec 0.604, Tokens/sec 9.664, Trained Tokens 160, Peak mem 5.730 GB
Iter 20: Train loss 0.172, Learning Rate 2.000e-04, It/sec 0.614, Tokens/sec 9.829, Trained Tokens 320, Peak mem 5.730 GB
Iter 30: Train loss 0.090, Learning Rate 2.000e-04, It/sec 0.614, Tokens/sec 9.823, Trained Tokens 480, Peak mem 5.730 GB
Iter 40: Train loss 0.557, Learning Rate 2.000e-04, It/sec 0.615, Tokens/sec 9.834, Trained Tokens 640, Peak mem 5.730 GB
Iter 50: Train loss 0.113, Learning Rate 2.000e-04, It/sec 0.615, 

In [ ]:
# ── Load model with T2A_demo_only adapter ──────────────────────────────
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w2/T2A_demo_only",
)

# ── Inference ──────────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_demo, return_probs=True
)

# ── Evaluate ───────────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T2A_demo_only")

# ── Calibration ────────────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T2A_demo_only",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T2A_demo_only.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ───────────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2A_demo_only.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2A_demo_only"] = elapsed
print(f"\n  ⏱ T2A_demo_only done in {elapsed:.1f}s")

# ── Clean up ───────────────────────────────────────────────────────
del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11703 [00:00<?, ?it/s]


── T2A_demo_only ──
  Label: T2A_demo_only
  N_test: 11703
  Accuracy: 0.8876
  Recall: 0.1306
  Precision: 0.746
  F1: 0.2222
  Unknown: 0
              precision    recall  f1-score   support

          No       0.89      0.99      0.94     10263
         Yes       0.75      0.13      0.22      1440

    accuracy                           0.89     11703
   macro avg       0.82      0.56      0.58     11703
weighted avg       0.87      0.89      0.85     11703

  ECE: 0.1070 | Saved → ./results_wave2/calibration_T2A_demo_only.png

  ⏱ T2A_demo_only done in 3750.9s


### Training and Inference for T2C_longitudinal_model

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T2C_longitudinal_model \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w2/T2C_longitudinal_model \
    --save-every 500 \
    -c ./configs/lora_config.yaml

In [ ]:
# ── Load model with T2C_longitudinal_model adapter ──────────────────────────────
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w2/T2C_longitudinal_model",
)

# ── Inference ──────────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_longitudinal, return_probs=True
)

# ── Evaluate ───────────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T2C_longitudinal_model")

# ── Calibration ────────────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T2C_longitudinal_model",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T2C_longitudinal_model.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ───────────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2C_longitudinal_model.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2C_longitudinal_model"] = elapsed
print(f"\n  ⏱ T2C_longitudinal_model done in {elapsed:.1f}s")

# ── Clean up ───────────────────────────────────────────────────────
del model_mlx, tok_mlx

### Inference for zero shot

In [ ]:
## Inference for zero shot
t0 = time.time()

model_mlx, tok_mlx = load(
    MODEL_ID,
    # no adapter_path — base model only
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3A_zeroshot")

ece = calibration_analysis(
    probs, refs,
    label_name="T3A_zeroshot",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T3A_zeroshot.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3A_zeroshot.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3A_zeroshot"] = elapsed
print(f"\n  ⏱ T3A_zeroshot done in {elapsed:.1f}s")

del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11703 [00:00<?, ?it/s]


── T3A_zeroshot ──
  Label: T3A_zeroshot
  N_test: 11703
  Accuracy: 0.6004
  Recall: 0.2222
  Precision: 0.0826
  F1: 0.1204
  Unknown: 0
              precision    recall  f1-score   support

          No       0.86      0.65      0.74     10263
         Yes       0.08      0.22      0.12      1440

    accuracy                           0.60     11703
   macro avg       0.47      0.44      0.43     11703
weighted avg       0.76      0.60      0.67     11703

  ECE: 0.0957 | Saved → ./results_wave2/calibration_T3A_zeroshot.png

  ⏱ T3A_zeroshot done in 3236.5s


### Training and Inference for T3A_narrative

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T3A_narrative \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w2/T3A_narrative \
    --save-every 500 \
    -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 47127.01it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:19<00:00,  1.31it/s]
Iter 1: Val loss 7.380, Val took 19.126s
Iter 10: Train loss 3.609, Learning Rate 2.000e-04, It/sec 0.660, Tokens/sec 10.561, Trained Tokens 160, Peak mem 5.275 GB
Iter 20: Train loss 0.133, Learning Rate 2.000e-04, It/sec 0.656, Tokens/sec 10.502, Trained Tokens 320, Peak mem 5.784 GB
Iter 30: Train loss 0.072, Learning Rate 2.000e-04, It/sec 0.683, Tokens/sec 10.925, Trained Tokens 480, Peak mem 5.784 GB
Iter 40: Train loss 0.284, Learning Rate 2.000e-04, It/sec 0.653, Tokens/sec 10.444, Trained Tokens 640, Peak mem 5.784 GB
Iter 50: Train loss 1.474, Learning Rate 2.000e-04, It/sec 0.6

In [ ]:
# ── Load model with T3A_narrative adapter ──────────────────────────────
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w2/T3A_narrative",
)

# ── Inference ──────────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_narrative_full, return_probs=True
)

# ── Evaluate ───────────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T3A_narrative")

# ── Calibration ────────────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T3A_narrative",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T3A_narrative.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ───────────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3A_narrative.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3A_narrative"] = elapsed
print(f"\n  ⏱ T3A_narrative done in {elapsed:.1f}s")

# ── Clean up ───────────────────────────────────────────────────────
del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11703 [00:00<?, ?it/s]


── T3A_narrative ──
  Label: T3A_narrative
  N_test: 11703
  Accuracy: 0.9551
  Recall: 0.7875
  Precision: 0.8375
  F1: 0.8117
  Unknown: 0
              precision    recall  f1-score   support

          No       0.97      0.98      0.97     10263
         Yes       0.84      0.79      0.81      1440

    accuracy                           0.96     11703
   macro avg       0.90      0.88      0.89     11703
weighted avg       0.95      0.96      0.95     11703

  ECE: 0.0261 | Saved → ./results_wave2/calibration_T3A_narrative.png

  ⏱ T3A_narrative done in 4058.5s


### Training and Inference for T3B_minimal

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T3B_minimal \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w2/T3B_minimal \
    --save-every 500 \
    -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 19650.57it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:18<00:00,  1.35it/s]
Iter 1: Val loss 7.367, Val took 18.585s
Iter 10: Train loss 3.629, Learning Rate 2.000e-04, It/sec 0.686, Tokens/sec 10.971, Trained Tokens 160, Peak mem 5.275 GB
Iter 20: Train loss 0.197, Learning Rate 2.000e-04, It/sec 0.702, Tokens/sec 11.235, Trained Tokens 320, Peak mem 5.275 GB
Iter 30: Train loss 0.052, Learning Rate 2.000e-04, It/sec 0.560, Tokens/sec 8.963, Trained Tokens 480, Peak mem 5.275 GB
Iter 40: Train loss 0.234, Learning Rate 2.000e-04, It/sec 0.535, Tokens/sec 8.560, Trained Tokens 640, Peak mem 5.275 GB
Iter 50: Train loss 0.407, Learning Rate 2.000e-04, It/sec 0.511

In [ ]:
# ── Load model with T3B_minimal adapter ──────────────────────────────
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w2/T3B_minimal",
)

# ── Inference ──────────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_minimal, return_probs=True
)

# ── Evaluate ───────────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T3B_minimal")

# ── Calibration ────────────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T3B_minimal",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T3B_minimal.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ───────────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3B_minimal.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3B_minimal"] = elapsed
print(f"\n  ⏱ T3B_minimal done in {elapsed:.1f}s")

# ── Clean up ───────────────────────────────────────────────────────
del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11703 [00:00<?, ?it/s]


── T3B_minimal ──
  Label: T3B_minimal
  N_test: 11703
  Accuracy: 0.9551
  Recall: 0.7875
  Precision: 0.8375
  F1: 0.8117
  Unknown: 0
              precision    recall  f1-score   support

          No       0.97      0.98      0.97     10263
         Yes       0.84      0.79      0.81      1440

    accuracy                           0.96     11703
   macro avg       0.90      0.88      0.89     11703
weighted avg       0.95      0.96      0.95     11703

  ECE: 0.0261 | Saved → ./results_wave2/calibration_T3B_minimal.png

  ⏱ T3B_minimal done in 3581.5s


### Save Results Summary

In [ ]:
results_df = pd.DataFrame(all_metrics)
results_csv = os.path.join(OUTPUT_DIR, "results_wave2_summary.csv")
results_df.to_csv(results_csv, index=False)
print("\n── Final Results (Wave 2) ──")
print(results_df.to_string(index=False))
print(f"\nSaved → {results_csv}")

print("\n── Experiment Runtimes ──")
for exp, t in experiment_times.items():
    print(f"  {exp}: {t:.1f}s")


── Final Results (Wave 2) ──
          Label  N_test  Accuracy  Recall  Precision     F1  Unknown    ECE
 T1A_imbalanced   11703    0.8876  0.1306     0.7460 0.2222        0 0.1070
T1B_oversampled   11703    0.8629  0.6451     0.4594 0.5367        0 0.0437
  T2A_demo_only   11703    0.8876  0.1306     0.7460 0.2222        0 0.1070
   T3A_zeroshot   11703    0.6004  0.2222     0.0826 0.1204        0 0.0957
  T3A_narrative   11703    0.9551  0.7875     0.8375 0.8117        0 0.0261
    T3B_minimal   11703    0.9551  0.7875     0.8375 0.8117        0 0.0261

Saved → ./results_wave2/results_wave2_summary.csv

── Experiment Runtimes ──
  T1A_imbalanced: 9778.6s
  T1B_oversampled: 3753.1s
  T2A_demo_only: 3750.9s
  T3A_zeroshot: 3236.5s
  T3A_narrative: 4058.5s
  T3B_minimal: 3581.5s
